## SE(3)-Transformer code of Weiss et al.

- Source: https://gitlab.com/benhalpe/PBHs-interpretability
- Citation: Weiss, T., Wahab, A., Bronstein, A. M. & Gershoni-Poranne, R. Interpretable Deep-Learning Unveils Structure–Property Relationships in Polybenzenoid Hydrocarbons. *J. Org. Chem.* **88**, 9645–9656 (2023).

Setup: download the repository, rename the repository folder to `pbh`, and put it in the root of this repository. The notebook imports `pbh.utils.args`, `pbh.data.aromatic_dataloader`, and `pbh.se3_transformer.models`, loads `data/extras/compas1_modified_subset.csv`, and loads the xyz files of COMPAS-1D from `data/compas1/b3lyp_8678/`. Extract the xyz files and install the dependencies:

```
tar xzf data/compas1/compas-1D_b3lyp_def2-svp.tar.gz -C data/compas1/
pip install git+https://github.com/AMLab-Amsterdam/lie_learn.git
pip install torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
pip install dgl==2.4.0+cu121 -f https://data.dgl.ai/wheels/torch-2.4/cu121/repo.html
pip install numpy lightning rdkit pandas scikit-learn networkx scipy tensorboard tqdm matplotlib cclib
```

In [1]:
import gc
import os

import dgl
import numpy as np
import pandas as pd
import torch

if os.path.basename(os.getcwd()) != "pbh":
    os.chdir("../pbh")
import warnings

from lightning import pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger
from pbh.data.aromatic_dataloader import AromaticDataset
from pbh.se3_transformer.models import create_model
from pbh.utils.args import Args
from sklearn.model_selection import StratifiedKFold, train_test_split
from torch.utils.data import DataLoader

from benchmarks import LightningModule, MultiTargetScaler, PrintEpochMetricsCallback

warnings.filterwarnings("ignore", category=FutureWarning, module="dgl")
warnings.filterwarnings(
    "ignore",
    message=".*You are using `torch.load` with `weights_only=False`.*",
    category=FutureWarning,
)

torch.set_float32_matmul_precision("medium")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

K_FOLDS = 5
TARGETS = ["HOMO_eV", "LUMO_eV", "GAP_eV"]
SUBSET = "../data/extras/compas1_modified_subset.csv"
XYZ_ROOT = "../data/compas1/b3lyp_8678/"
GRAPH = "rings"  # 'rings' = GOR (Weiss headline). 'atoms' = GOA

# Same stratified folds as compas1_pac_matrix_gcnn.ipynb
df = pd.read_csv(SUBSET).dropna(subset=TARGETS).reset_index(drop=True)
print("Modified COMPAS-1D subset:", len(df), "molecules")

counts = df["n_rings"].value_counts()
rare = counts[counts < 5].index.tolist()
df["stratify_group"] = df["n_rings"].replace(rare, "rare").astype(str)
if 0 < (df["stratify_group"] == "rare").sum() < 5:
    df.loc[df["stratify_group"] == "rare", "stratify_group"] = str(counts.idxmax())

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
model_path_template = (
    "../models/se3_transformer_3.9M-graph_of_rings-[COMPAS-1D]-fold_{fold}"
)
model_logs_dir = "../logs"
model_dataset_dir = "../dataset"
model_logs_template = "se3_transformer_3.9M-graph_of_rings-[COMPAS-1D]-fold_{fold}"


def base_args():
    a = Args().parse_args([])
    a.csv_file, a.xyz_root = SUBSET, XYZ_ROOT
    a.target_features = ",".join(TARGETS)  # 3 targets -> SE3 out_dim = 3
    a.rings_graph = GRAPH == "rings"
    a.normalize = False  # Scaling handled by MultiTargetScaler below
    a.transform = False
    a.sample_rate = 1.0
    a.device = device
    return a


def make_datasets(a, tr_df, va_df, te_df, all_df):
    a.train_df, a.val_df, a.test_df, a.df_all = tr_df, va_df, te_df, all_df
    return (
        AromaticDataset(a, "train"),
        AromaticDataset(a, "val"),
        AromaticDataset(a, "test"),
    )


def make_collate(scaler):
    def _collate(samples):
        graphs, ys = zip(*samples)
        g = dgl.batch(graphs)
        y = scaler.transform(torch.cat(ys, dim=0).float())
        return g, y

    return _collate


class SE3LightningModule(LightningModule):
    # Lightning cannot transfer DGL graphs to device
    def transfer_batch_to_device(self, batch, device, dataloader_idx=0):
        g, y = batch
        return g.to(device), y.to(device)

ModuleNotFoundError: No module named 'dgl'

In [ ]:
fold_results = []

for fold, (tv_idx, te_idx) in enumerate(skf.split(df, df["stratify_group"]), 1):
    if fold in [1, 2]:
        pass
    else:
        continue
    print(f"\n======[ Fold {fold}/{K_FOLDS} ]======")

    tr_idx, va_idx = train_test_split(
        tv_idx,
        test_size=0.125,
        random_state=42,
        stratify=df.loc[tv_idx, "stratify_group"],
    )
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    te_df = df.iloc[te_idx].reset_index(drop=True)

    a = base_args()
    train_dataset, val_dataset, test_dataset = make_datasets(a, tr_df, va_df, te_df, df)

    scaler = MultiTargetScaler()
    scaler.fit(tr_df[TARGETS].values.astype("float32"))
    coll = make_collate(scaler)

    train_loader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        collate_fn=coll,
        num_workers=2,
        pin_memory=False,
        persistent_workers=False,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        collate_fn=coll,
        num_workers=2,
        pin_memory=False,
        persistent_workers=False,
    )

    model = create_model(a, train_dataset)
    lightning_model = SE3LightningModule(model, TARGETS, scaler, lr=1e-3)

    save_path_dir = model_path_template.format(fold=fold)
    os.makedirs(save_path_dir, exist_ok=True)

    checkpoint_callback = ModelCheckpoint(
        dirpath=save_path_dir,
        filename="best-{epoch}-{val_loss:.5f}",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    )

    logger = TensorBoardLogger(
        save_dir=model_logs_dir, name=model_logs_template.format(fold=fold)
    )

    trainer = pl.Trainer(
        max_epochs=100,
        gradient_clip_val=1.0,
        gradient_clip_algorithm="norm",
        accelerator="auto",
        logger=logger,
        devices=1,
        callbacks=[checkpoint_callback, PrintEpochMetricsCallback(TARGETS)],
        enable_checkpointing=True,
        num_sanity_val_steps=0,
        log_every_n_steps=1,
        precision="bf16-mixed" if device.type == "cuda" else "32-true",
        enable_progress_bar=False,
    )
    trainer.fit(lightning_model, train_loader, val_loader)

    best_model_score = checkpoint_callback.best_model_score
    if best_model_score is None:
        raise RuntimeError("The checkpoint callback did not record a validation loss.")
    best_val_loss = best_model_score.item()
    print(f"Best scaled validation loss for fold {fold}: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)

    best_model_path = checkpoint_callback.best_model_path
    best_lightning_model = SE3LightningModule.load_from_checkpoint(
        best_model_path,
        model=model,
        property_names=TARGETS,
        scaler=scaler,
        enable_progress_bar=False,
    )
    torch.save(best_lightning_model.model.state_dict(), save_path_dir + "/best.ckpt")

    lightning_model.cpu()
    best_lightning_model.cpu()

    del (
        train_dataset,
        val_dataset,
        test_dataset,
        train_loader,
        val_loader,
        model,
        lightning_model,
        best_lightning_model,
    )
    del trainer, checkpoint_callback, logger

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ k-fold results ]======")
print(f"Scaled Loss per fold: {[round(float(x), 4) for x in fold_results]}")
print(f"Average Scaled Loss: {np.mean(fold_results):.5f} ± {np.std(fold_results):.5f}")

In [ ]:
metric_keys = ["mae", "rmsd", "max", "min", "r2"]
all_results = {
    "train": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in TARGETS
    },
    "test": {
        property_name: {metric: [] for metric in metric_keys}
        for property_name in TARGETS
    },
}

for fold, (tv_idx, te_idx) in enumerate(skf.split(df, df["stratify_group"]), 1):
    print(f"\n======[ Evaluating fold {fold}/{K_FOLDS} ]======")

    tr_idx, va_idx = train_test_split(
        tv_idx,
        test_size=0.125,
        random_state=42,
        stratify=df.loc[tv_idx, "stratify_group"],
    )
    tr_df = df.iloc[tr_idx].reset_index(drop=True)
    va_df = df.iloc[va_idx].reset_index(drop=True)
    te_df = df.iloc[te_idx].reset_index(drop=True)

    a = base_args()
    train_ds, val_ds, test_ds = make_datasets(a, tr_df, va_df, te_df, df)

    scaler = MultiTargetScaler()
    scaler.fit(tr_df[TARGETS].values.astype("float32"))
    coll = make_collate(scaler)

    train_loader = DataLoader(
        train_ds,
        batch_size=256,
        shuffle=False,
        collate_fn=coll,
        num_workers=2,
        pin_memory=False,
        persistent_workers=False,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=256,
        shuffle=False,
        collate_fn=coll,
        num_workers=2,
        pin_memory=False,
        persistent_workers=False,
    )

    core_model = create_model(a, train_ds)
    load_path = model_path_template.format(fold=fold) + "/best.ckpt"
    core_model.load_state_dict(torch.load(load_path, map_location=device))

    lightning_model = SE3LightningModule(core_model, TARGETS, scaler)

    trainer = pl.Trainer(
        accelerator="auto", devices=1, logger=False, enable_progress_bar=False
    )

    print("\n======[ Train Set Evaluation ]======")
    train_results = trainer.test(
        lightning_model, dataloaders=train_loader, verbose=False
    )[0]

    print("\n======[ Test Set Evaluation ]======")
    test_results = trainer.test(
        lightning_model, dataloaders=test_loader, verbose=False
    )[0]

    for i, prop in enumerate(TARGETS):
        print(f"======[ {prop} ]======")
        t_mae = train_results[f"test_mae_{prop}"]
        t_rmsd = train_results[f"test_rmsd_{prop}"]
        t_max = train_results[f"test_max_{prop}"]
        t_min = train_results[f"test_min_{prop}"]
        t_r2 = train_results[f"test_r2_{prop}"]
        print(
            f" + Train -> MAE: {t_mae:.4f}\tRMSD: {t_rmsd:.4f}\tMax: {t_max:.4f}\tMin: {t_min:.4f}\tR^2: {t_r2:.4f}"
        )

        te_mae = test_results[f"test_mae_{prop}"]
        te_rmsd = test_results[f"test_rmsd_{prop}"]
        te_max = test_results[f"test_max_{prop}"]
        te_min = test_results[f"test_min_{prop}"]
        te_r2 = test_results[f"test_r2_{prop}"]
        print(
            f" + Test -> MAE: {te_mae:.4f}\tRMSD: {te_rmsd:.4f}\tMax: {te_max:.4f}\tMin: {te_min:.4f}\tR^2: {te_r2:.4f}"
        )

        for metric in metric_keys:
            all_results["train"][prop][metric].append(
                train_results[f"test_{metric}_{prop}"]
            )
            all_results["test"][prop][metric].append(
                test_results[f"test_{metric}_{prop}"]
            )

    del core_model, lightning_model, trainer, scaler
    del train_results, test_results
    del train_loader, test_loader, train_ds, val_ds, test_ds
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n======[ Cross-validation results (mean ± std) ]======")

for property_name in TARGETS:
    print(f"\n======[ {property_name} ]======")

    train_statistics = {
        metric: (
            np.mean(all_results["train"][property_name][metric]),
            np.std(all_results["train"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Train -> "
        f"MAE: {train_statistics['mae'][0]:.4f} ± {train_statistics['mae'][1]:.4f}\t"
        f"RMSD: {train_statistics['rmsd'][0]:.4f} ± {train_statistics['rmsd'][1]:.4f}\t"
        f"Max: {train_statistics['max'][0]:.4f} ± {train_statistics['max'][1]:.4f}\t"
        f"Min: {train_statistics['min'][0]:.4f} ± {train_statistics['min'][1]:.4f}\t"
        f"R^2: {train_statistics['r2'][0]:.4f} ± {train_statistics['r2'][1]:.4f}"
    )

    test_statistics = {
        metric: (
            np.mean(all_results["test"][property_name][metric]),
            np.std(all_results["test"][property_name][metric], ddof=1),
        )
        for metric in metric_keys
    }

    print(
        f" + Test -> "
        f"MAE: {test_statistics['mae'][0]:.4f} ± {test_statistics['mae'][1]:.4f}\t"
        f"RMSD: {test_statistics['rmsd'][0]:.4f} ± {test_statistics['rmsd'][1]:.4f}\t"
        f"Max: {test_statistics['max'][0]:.4f} ± {test_statistics['max'][1]:.4f}\t"
        f"Min: {test_statistics['min'][0]:.4f} ± {test_statistics['min'][1]:.4f}\t"
        f"R^2: {test_statistics['r2'][0]:.4f} ± {test_statistics['r2'][1]:.4f}"
    )